# U.S. Retail Sales Forecasting

## Project Overview
Analyze monthly U.S. retail sales (January 1992–June 2021), establish a linear-trend forecasting baseline, and evaluate predictions on the final 12 months.


## 1. Load and plot the data
- The dataset is in wide format, where each row represents one year. Each column represents a month (JAN, FEB, MAR, etc.)
- Wide format makes it difficult to create a continuous timeline, plot properly, and build regression-based forecasting models.
- For time series modeling, long format is needed, where each row represents a single month and includes a proper Date column
   - Time series models rely on chronological ordering.
 - Melt() is used to convert wide format to long format. As a result, the dataset supports proper time ordering, trend analysis, regression modeling, and accurate visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from IPython.display import display, HTML

display(HTML("<h3 style='font-weight:bold;'>1. Load the retail Dataset & Plot the data</h3>"))

from pathlib import Path

data_path = Path("data/raw/us_retail_sales.csv")
if not data_path.exists():
    data_path = Path("../data/raw/us_retail_sales.csv")
df = pd.read_csv(data_path)
display(HTML("<b>First 5 Rows of the Dataset:</b>"))
display(df.head())

df_long = df.melt(
    id_vars=["YEAR"],           
    var_name="Month",           
    value_name="Retail_Sales")
df_long["Date"] = pd.to_datetime(
    df_long["YEAR"].astype(str) + "-" + df_long["Month"],
    format="%Y-%b"
)
df_long = df_long.sort_values("Date")
df_long = df_long[["Date", "Retail_Sales"]]

df_display = df_long.copy()
df_display.columns = ["Date", "Retail Sales"]
display(HTML("<b>Reshaped Monthly Time Series (first 5 rows):</b>"))
display(df_display.head())

display(HTML("<b>Graph:</b>"))
plt.figure(figsize=(9,5))
plt.plot(df_long["Date"], df_long["Retail_Sales"])
plt.title("US Monthly Retail Sales Over Time (Jan 1992-Jun 2021) ")
plt.xlabel("Date")
plt.ylabel("Retail Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### **Observations:**
1. Retail sales show a strong long-term upward trend from 1992 to 2019.
2. Clear seasonal patterns repeat every year (likely holiday effects).
3. A noticeable dip appears during the 2008 financial crisis.
4. A sharp drop occurs in 2020 due to COVID-19, followed by a rapid recovery.
5. Overall, the data exhibits trend, seasonality, and structural shocks.

## 2. Split this data into a training and test set
Why a chronological train test split?
- Time series must never be randomly split. Random splitting would leak future information.
- Simulate real forecasting:
    - Test on unseen future values (Use the last year of data (July 2020 – June 2021) of data as test set)
    - Train using historical data (all remaining data)

In [ ]:
display(HTML("<h3 style='font-weight:bold;'>2. Split data into a training and test set </h3>"))

df_long.set_index("Date", inplace=True)
train = df_long[df_long.index < "2020-07-01"]
test = df_long[
    (df_long.index >= "2020-07-01") &
    (df_long.index <= "2021-06-30")]

display(HTML("<br>"))
display(HTML(f"<b>Training set size:</b> {len(train)}"))
display(HTML(f"<b>Test set size:</b> {len(test)}"))

#### **Observations:**
- Model trained on 342 months (historical data).
- Tested on 12 months (post-COVID recovery period).
- Makes evaluation realistic and more challenging.

### 3. Build a predictive model for the monthly retail sales using training data
- Since this is time-series data, a numeric time index is created.
- Linear Regression model requires numerical input features; dates cannot be used directly.
- The Time_Index variable captures the progression of time, enabling the model to learn and represent the underlying linear trend.

In [ ]:
display(HTML("<h3 style='font-weight:bold;'>3. Build Predictive Model using train data</h3>"))

train = train.copy()
test = test.copy()

train["Time_Index"] = np.arange(len(train))
test["Time_Index"] = np.arange(len(train), len(train) + len(test))

X_train = train[["Time_Index"]]
y_train = train["Retail_Sales"]

model = LinearRegression()
model.fit(X_train, y_train)

display(HTML("<br>"))
display(HTML("<b>Model Parameters:</b>")) 
display(HTML(f"<b>Intercept:</b> {round(model.intercept_,2)}"))
display(HTML(f"<b>Slope (monthly growth rate):</b> {round(model.coef_[0],2)}"))

#### **Observations:**
- The intercept (150875.54) represents the estimated baseline value at the beginning of the time series, serving as the model’s starting level.
- The slope (873.05) indicates a positive monthly growth rate, with the value increasing by approximately 873 units per month on average.
- Model successfully captures the overall long-term growth trend.
- Limitation - Model does not account for seasonality, structural breaks, or sudden economic shocks.

In [ ]:
display(HTML("<h3 style='font-weight:bold;'>4. Predict monthly retail sales on test period</h3>"))

X_test = test[["Time_Index"]]
y_test = test["Retail_Sales"]

predictions = model.predict(X_test)

display(HTML("<br>"))
display(HTML("<b>Graph:</b>"))
plt.figure()
plt.plot(test.index, y_test, label="Actual Sales")
plt.plot(test.index, predictions, label="Predicted Sales")
plt.title("Actual vs Predicted Retail Sales (Test data)")
plt.xlabel("Date")
plt.ylabel("Retail Sales")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
}).round(2)

display(HTML("<b>First 5 Actual vs Predicted (Rounded to 2 Decimals):</b>"))
display(results.head())

#### **Observations:**
- Predicted values follow a smooth upward line.
- Actual values fluctuate due to below factors:
    - Seasonality
    - Post-COVID recovery spike
- The model underestimates rapid recovery growth.

### 5. Report the RMSE of the model predictions on the test set
- RMSE measures prediction error in same units as sales.

In [ ]:
display(HTML("<h3 style='font-weight:bold;'>5. Report the RMSE of the model predictions on the test set</h3>"))

rmse = np.sqrt(mean_squared_error(y_test, predictions))

display(HTML("<br>"))
display(HTML(f"<b>Root Mean Squared Error (RMSE):</b> {round(rmse,2)}"))

#### **Observations:**
- Average prediction error is 66433 units.
- Error is relatively high because:
    - Model ignores seasonality.
    - The model cannot capture the structural break (COVID shock).
- This is a baseline trend model only.

### **Summary of Findings:**
- From 1992 to 2019, US retail sales show a clear and steady long-term increase, indicating ongoing economic growth and higher consumer spending over time.
- The data displays noticeable seasonal patterns each year, likely influenced by holiday shopping.
- Large disruptions are evident during major events, such as the 2008 financial crisis and the sharp decline caused by COVID-19 in 2020, followed by a quick rebound afterward.
- The linear regression model successfully captures the overall upward trend but fails to account for seasonal patterns and sudden economic shocks, leading to relatively high prediction error (RMSE ~ 66433), especially during volatile periods.

### **Conclusion:**
US retail sales show a consistent long-term upward trend, noticeable seasonal patterns, and occasional sharp economic shocks. Although linear regression can model the general growth trend, it falls short for accurate forecasting because it overlooks seasonal fluctuations and sudden disruptions. To achieve more reliable predictions, advanced time series models that account for both seasonality and unexpected economic events are necessary.